# Кто дал оценку

Финальное решение задачи идентификации автора оценки перевода. Полный пайплайн объединяет статистические и текстовые признаки, ансамбль классических моделей, COMETinho и специальную обработку пар строк.

**Leaderboard ROC-AUC: 0.7622.**


## 1. Зависимости и данные

Положите `train.csv` и `test.csv` рядом с ноутбуком. Для повторного COMET-инференса используется кэш с проверкой хеша входных данных.


In [ ]:
# Если библиотек нет в окружении, установим их автоматически.
import importlib.util
import subprocess
import sys

packages = {
    'lightgbm': 'lightgbm',
    'comet': 'unbabel-comet',
    'pyarrow': 'pyarrow',
}
missing = [pip_name for module, pip_name in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Dependencies are ready')


In [ ]:
import hashlib
import json
import re
import time
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from scipy.special import expit
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 20260920
ALPHA = 0.25  # проверенный на leaderboard вес, давший 76.22

train_path = Path('train.csv')
if not train_path.exists():
    train_path = Path('train (1).csv')
test_path = Path('test.csv')
if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError('Положите train.csv и test.csv рядом с блокнотом')

train = pd.read_csv(train_path).reset_index(drop=True)
test = pd.read_csv(test_path).reset_index(drop=True)
target = (train['annotator'] == 'person2').astype(int).to_numpy()
assert train['annotator'].isin(['person1', 'person2']).all()
print('train:', train.shape, 'test:', test.shape, 'target mean:', target.mean())


## 2. Признаки и парная структура


In [ ]:
def tokens(value):
    return re.findall(r'[\w]+', str(value).lower(), re.UNICODE)

def ngrams(value, n):
    value = ' '.join(tokens(value))
    return Counter(value[i:i+n] for i in range(max(0, len(value) - n + 1)))

def dice(left, right, n):
    left_counts, right_counts = ngrams(left, n), ngrams(right, n)
    denominator = sum(left_counts.values()) + sum(right_counts.values())
    return 2 * sum((left_counts & right_counts).values()) / denominator if denominator else 1.0

def jaccard(left, right):
    left_tokens, right_tokens = set(tokens(left)), set(tokens(right))
    union = left_tokens | right_tokens
    return len(left_tokens & right_tokens) / len(union) if union else 1.0

def make_features(frame):
    features = {}
    for column in ['direction', 'preference']:
        features[column] = frame[column].fillna('NA').astype(str)
    features['rating_a_cat'] = frame.rating_a.fillna(-1).astype(str)
    features['rating_b_cat'] = frame.rating_b.fillna(-1).astype(str)
    features['decision'] = features['preference'] + '_' + features['rating_a_cat'] + '_' + features['rating_b_cat']
    features['direction_decision'] = features['direction'] + '_' + features['decision']
    features['rating_a'] = frame.rating_a
    features['rating_b'] = frame.rating_b
    features['rating_sum'] = frame.rating_a + frame.rating_b
    features['rating_diff'] = frame.rating_a - frame.rating_b

    text_columns = ['original', 'reference', 'translation_1', 'translation_2']
    text_stats = {}
    for column in text_columns:
        values = frame[column].fillna('').astype(str)
        stats = {
            'chars': values.str.len(),
            'tokens': values.map(lambda value: len(tokens(value))),
            'digits': values.str.count(r'\d'),
            'punct': values.str.count(r'[^\w\s]'),
            'upper': values.str.count(r'[A-ZА-ЯӘҒҚҢӨҰҮҺІ]'),
            'kazakh': values.str.count(r'[әғқңөұүһіӘҒҚҢӨҰҮҺІ]'),
            'cyrillic': values.str.count(r'[А-Яа-яӘҒҚҢӨҰҮҺІәғқңөұүһі]'),
        }
        text_stats[column] = stats
        for name, stat_values in stats.items():
            features[f'{column}_{name}'] = stat_values

    pairs = [
        ('reference', 'translation_1', 'ref_a'),
        ('reference', 'translation_2', 'ref_b'),
        ('translation_1', 'translation_2', 'a_b'),
        ('original', 'reference', 'source_ref'),
    ]
    for left_column, right_column, prefix in pairs:
        left = frame[left_column].fillna('').astype(str)
        right = frame[right_column].fillna('').astype(str)
        features[f'{prefix}_sequence'] = [
            SequenceMatcher(None, a.lower(), b.lower(), autojunk=False).ratio() for a, b in zip(left, right)
        ]
        features[f'{prefix}_jaccard'] = [jaccard(a, b) for a, b in zip(left, right)]
        for n in (1, 2, 3, 4):
            features[f'{prefix}_dice_{n}'] = [dice(a, b, n) for a, b in zip(left, right)]

    for stat in ('chars', 'tokens', 'digits', 'punct', 'upper', 'kazakh', 'cyrillic'):
        a, b, ref = text_stats['translation_1'][stat], text_stats['translation_2'][stat], text_stats['reference'][stat]
        features[f'translations_{stat}_difference'] = a - b
        features[f'translations_{stat}_sum'] = a + b
        features[f'a_ref_{stat}_ratio'] = a / (ref + 1)
        features[f'b_ref_{stat}_ratio'] = b / (ref + 1)

    for metric in ('sequence', 'jaccard', 'dice_1', 'dice_2', 'dice_3', 'dice_4'):
        features[f'similarity_{metric}_difference'] = np.asarray(features[f'ref_a_{metric}']) - np.asarray(features[f'ref_b_{metric}'])
        features[f'similarity_{metric}_sum'] = np.asarray(features[f'ref_a_{metric}']) + np.asarray(features[f'ref_b_{metric}'])
    return pd.DataFrame(features, index=frame.index).copy()

def enforce_pairs(probabilities, frame, weight=1.0):
    adjusted = probabilities.copy()
    positions = pd.Series(np.arange(len(frame)), index=frame.index)
    for labels in frame.groupby('original').groups.values():
        if len(labels) != 2:
            continue
        first, second = positions.loc[list(labels)].to_numpy()
        difference = probabilities[first] - probabilities[second]
        adjusted[first] = 0.5 + weight * difference / 2
        adjusted[second] = 0.5 - weight * difference / 2
    return np.clip(adjusted, 0, 1)


In [ ]:
# Общие handcrafted-признаки.
all_frame = pd.concat([train.drop(columns='annotator'), test], ignore_index=True)
all_base_features = make_features(all_frame)
train_features = all_base_features.iloc[:len(train)].reset_index(drop=True)
test_features = all_base_features.iloc[len(train):].reset_index(drop=True)
categorical = [column for column in train_features if train_features[column].dtype == 'object']
numeric = [column for column in train_features if column not in categorical]

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
splits = list(splitter.split(train_features, target, groups=train.original))

encoded = pd.get_dummies(
    pd.concat([train_features, test_features], ignore_index=True),
    columns=categorical, dummy_na=True,
).replace([np.inf, -np.inf], np.nan)
encoded = encoded.fillna(encoded.iloc[:len(train)].median())
encoded_train = encoded.iloc[:len(train)]
encoded_test = encoded.iloc[len(train):]
print('base features:', train_features.shape)


## 3. Базовый ансамбль


In [ ]:
# Базовый ансамбль: те же параметры, которые использовались в submission 75.95.
logistic_test, extra_test, lgb_test = [], [], []
oof_logistic = np.zeros(len(train))
oof_extra = np.zeros(len(train))
oof_lgb = np.zeros(len(train))

for fold, (train_index, valid_index) in enumerate(splits):
    preprocessor = ColumnTransformer([
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical),
        ('numeric', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
    ])
    logistic = make_pipeline(
        preprocessor,
        LogisticRegression(C=1.0, max_iter=3000, random_state=RANDOM_STATE + fold),
    )
    logistic.fit(train_features.iloc[train_index], target[train_index])
    oof_logistic[valid_index] = logistic.predict_proba(train_features.iloc[valid_index])[:, 1]
    logistic_test.append(logistic.predict_proba(test_features)[:, 1])

    extra = ExtraTreesClassifier(
        n_estimators=600, min_samples_leaf=20, max_features=0.8,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE + fold,
    )
    extra.fit(encoded_train.iloc[train_index], target[train_index])
    oof_extra[valid_index] = extra.predict_proba(encoded_train.iloc[valid_index])[:, 1]
    extra_test.append(extra.predict_proba(encoded_test)[:, 1])

    lgb = LGBMClassifier(
        n_estimators=500, learning_rate=0.025, num_leaves=5, min_child_samples=40,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=8, reg_alpha=1,
        verbosity=-1, random_state=RANDOM_STATE + fold,
    )
    lgb.fit(encoded_train.iloc[train_index], target[train_index])
    oof_lgb[valid_index] = lgb.predict_proba(encoded_train.iloc[valid_index])[:, 1]
    lgb_test.append(lgb.predict_proba(encoded_test)[:, 1])

base_oof = 0.4 * oof_logistic + 0.5 * oof_extra + 0.1 * oof_lgb
base_test = (
    0.4 * np.mean(logistic_test, axis=0)
    + 0.5 * np.mean(extra_test, axis=0)
    + 0.1 * np.mean(lgb_test, axis=0)
)
print('base row OOF AUC:', roc_auc_score(target, base_oof))


In [ ]:
# Коррекция пар строк с одинаковым original.
pair_preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('numeric', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
])
pair_train = pair_preprocessor.fit_transform(train_features)
pair_test = pair_preprocessor.transform(test_features)
if hasattr(pair_train, 'toarray'):
    pair_train = pair_train.toarray()
    pair_test = pair_test.toarray()

differences, pair_targets = [], []
for labels in train.groupby('original').groups.values():
    if len(labels) != 2:
        continue
    first, second = list(labels)
    difference = pair_train[first] - pair_train[second]
    differences.extend([difference, -difference])
    pair_targets.extend([target[first], target[second]])

pair_model = LogisticRegression(C=0.3, max_iter=3000, random_state=RANDOM_STATE)
pair_model.fit(np.asarray(differences), np.asarray(pair_targets))

baseline_score = base_test.copy()
for labels in test.groupby('original').groups.values():
    if len(labels) != 2:
        continue
    first, second = list(labels)
    direct = pair_model.predict_proba((pair_test[first] - pair_test[second]).reshape(1, -1))[0, 1]
    row_pair_first = 0.5 + (base_test[first] - base_test[second]) / 2
    combined_first = 0.1 * direct + 0.9 * row_pair_first
    baseline_score[first] = 0.5 + 2.5 * (combined_first - 0.5)
    baseline_score[second] = 1 - baseline_score[first]
print('pair correction is ready')


## 4. COMETinho и финальный blend


In [ ]:
# COMETinho-признаки. Первый запуск скачивает компактную модель с Hugging Face.
# Кэш parquet позволяет повторно запускать блокнот без нового инференса.
comet_cache = Path('comet_features.parquet')
comet_metadata = Path('comet_features.json')
fingerprint_columns = ['original', 'reference', 'translation_1', 'translation_2']
data_fingerprint = hashlib.sha256(
    pd.util.hash_pandas_object(all_frame[fingerprint_columns], index=True).values.tobytes()
).hexdigest()

cache_is_valid = False
if comet_cache.exists() and comet_metadata.exists():
    metadata = json.loads(comet_metadata.read_text(encoding='utf-8'))
    cache_is_valid = metadata.get('data_fingerprint') == data_fingerprint

if cache_is_valid:
    comet_features = pd.read_parquet(comet_cache)
    print('Loaded cached COMET features:', comet_features.shape)
else:
    from comet import download_model, load_from_checkpoint

    model_path = download_model('Unbabel/eamt22-cometinho-da')
    comet_model = load_from_checkpoint(model_path)
    records = []
    for column in ('translation_1', 'translation_2'):
        records.extend({
            'src': str(row.original),
            'mt': str(getattr(row, column)),
            'ref': str(row.reference),
        } for row in all_frame.itertuples(index=False))

    started = time.time()
    try:
        prediction = comet_model.predict(records, batch_size=32, gpus=1, num_workers=2)
    except (TypeError, RuntimeError):
        prediction = comet_model.predict(records, batch_size=32, gpus=0, num_workers=2)
    scores = np.asarray(prediction.scores)
    n = len(all_frame)
    comet_features = pd.DataFrame({'comet_a': scores[:n], 'comet_b': scores[n:]})
    comet_features['comet_diff'] = comet_features.comet_a - comet_features.comet_b
    comet_features['comet_sum'] = comet_features.comet_a + comet_features.comet_b
    comet_features['comet_min'] = comet_features[['comet_a', 'comet_b']].min(axis=1)
    comet_features['comet_max'] = comet_features[['comet_a', 'comet_b']].max(axis=1)
    comet_features.to_parquet(comet_cache, index=False)
    comet_metadata.write_text(
        json.dumps({'data_fingerprint': data_fingerprint}, indent=2),
        encoding='utf-8',
    )
    print('COMET features:', comet_features.shape, 'seconds:', round(time.time() - started, 1))


In [ ]:
def add_comet_features(base, frame, comet_frame):
    result = base.reset_index(drop=True).copy()
    comet_frame = comet_frame.reset_index(drop=True)
    for column in comet_frame:
        result[column] = comet_frame[column]

    diff = comet_frame['comet_diff']
    rating_diff = frame.rating_a.reset_index(drop=True) - frame.rating_b.reset_index(drop=True)
    preference = frame.preference.reset_index(drop=True)
    direction = frame.direction.reset_index(drop=True)
    preference_sign = preference.map({'a': 1.0, 'b': -1.0}).fillna(0.0)
    result['comet_preference_alignment'] = diff * preference_sign
    result['comet_rating_alignment'] = diff * rating_diff
    result['comet_rating_gap'] = diff - rating_diff / 2.0
    result['comet_rating_gap_abs'] = result['comet_rating_gap'].abs()
    result['comet_rank_agrees_preference'] = ((diff * preference_sign) > 0).astype(float)
    result['comet_rank_agrees_rating'] = ((diff * rating_diff) > 0).astype(float)
    result['comet_near_tie'] = (diff.abs() < 0.05).astype(float)
    for value in ('a', 'b', 'tie', 'both_bad'):
        result[f'comet_diff_pref_{value}'] = diff * (preference == value).astype(float)
    for value in ('kk->ru', 'ru->kk'):
        result[f'comet_diff_dir_{value}'] = diff * (direction == value).astype(float)
        result[f'comet_sum_dir_{value}'] = comet_frame.comet_sum * (direction == value).astype(float)
    for rating in (0.0, 1.0, 2.0):
        result[f'comet_a_rating_{int(rating)}'] = comet_frame.comet_a * (frame.rating_a.reset_index(drop=True) == rating).astype(float)
        result[f'comet_b_rating_{int(rating)}'] = comet_frame.comet_b * (frame.rating_b.reset_index(drop=True) == rating).astype(float)
    return result

all_comet_features = add_comet_features(all_base_features, all_frame, comet_features)
X = all_comet_features.iloc[:len(train)].reset_index(drop=True)
X_test = all_comet_features.iloc[len(train):].reset_index(drop=True)
comet_categorical = [column for column in X if X[column].dtype == 'object']
comet_numeric = [column for column in X if column not in comet_categorical]

comet_test_predictions = []
comet_oof = np.zeros(len(train))
for fold, (train_index, valid_index) in enumerate(splits):
    comet_preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), comet_categorical),
        ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), comet_numeric),
    ])
    model = make_pipeline(
        comet_preprocessor,
        LogisticRegression(C=0.03, max_iter=3000, random_state=RANDOM_STATE + fold),
    )
    model.fit(X.iloc[train_index], target[train_index])
    comet_oof[valid_index] = model.predict_proba(X.iloc[valid_index])[:, 1]
    comet_test_predictions.append(model.predict_proba(X_test)[:, 1])

comet_test = np.mean(comet_test_predictions, axis=0)
print('COMET-head OOF AUC:', roc_auc_score(target, comet_oof))


In [ ]:
# Финальная обработка COMET-оценок пар и проверенный blend alpha=0.25.
comet_final = comet_test.copy()
comet_pair = enforce_pairs(comet_test, test, weight=1.0)
paired = np.zeros(len(test), dtype=bool)
positions = pd.Series(np.arange(len(test)), index=test.index)
for labels in test.groupby('original').groups.values():
    if len(labels) == 2:
        paired[positions.loc[list(labels)].to_numpy()] = True
comet_final[paired] = 0.5 + 2.5 * (comet_pair[paired] - 0.5)

final_score = (1 - ALPHA) * baseline_score + ALPHA * comet_final
solution = pd.DataFrame({
    'id': test['id'].to_numpy(),
    'probability': expit(4.0 * (final_score - 0.5)),
})

assert len(solution) == len(test)
assert solution['id'].equals(test['id'])
assert np.isfinite(solution['probability']).all()
assert solution['probability'].between(0, 1, inclusive='both').all()
solution.to_csv('solution.csv', index=False)
print('Saved solution.csv:', solution.shape)
display(solution.head())
